# 02 - Cleaning, gap audit and feature engineering

Turns raw candles into a validated, segmented, feature-rich frame.

**Two rules drive everything here**

1. Every feature at row `t` uses rows `<= t` only. All primitives are
   backward-looking: `.shift(k>=1)`, trailing `.rolling()`, recursive `.ewm()`.
2. No absolute price level becomes a model input. BTC spans roughly $4k to
   >$100k over this period, so raw prices would let the model memorise the
   price regime instead of learning behaviour. Every price expression is a
   return, ratio, percentage distance or volatility.

**Gaps:** missing candles are never fabricated. Every discontinuity starts a new
*segment*, and later stages forbid a window or a target from crossing one.

In [ ]:
import sys, os, json
from pathlib import Path

# Make src/ importable whether this notebook runs from notebooks/ or the root.
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
print("project root:", ROOT)

In [ ]:
from preprocessing import preprocess, segment_lengths
from features import build_features, FEATURE_COLUMNS, FEATURE_GROUPS, feature_nan_report

clean, continuity = preprocess()
print("\n" + continuity.render())

In [ ]:
lens = segment_lengths(clean)
print("segments:", len(lens))
print(lens.describe())
if continuity.n_gaps == 0:
    print("\nNo gaps: the whole history is one contiguous segment.")

In [ ]:
import pandas as pd

featured = build_features(clean)
print("total features:", len(FEATURE_COLUMNS))

pd.set_option("display.max_colwidth", 200)
pd.DataFrame([{"group": g, "n": len(c), "columns": ", ".join(c)}
              for g, c in FEATURE_GROUPS.items()])

In [ ]:
# Feature NaNs are expected ONLY at the head, from indicator warmup.
nan = feature_nan_report(featured)
print(nan[nan > 0].to_string() if (nan > 0).any() else "no NaNs at all")
print("\nwarmup rows dropped by the pipeline: 500")
print("first fully-valid row index:", int(featured[FEATURE_COLUMNS].notna().all(axis=1).idxmax()))

In [ ]:
sample = ["return_1", "volatility_16", "rsi_14_norm", "bb_position",
          "taker_buy_ratio", "volume_zscore_96", "atr_pct_14"]
featured[sample].describe().T

In [ ]:
out = ROOT / "data" / "processed" / "btcusdt_15m_features.csv"
featured[["timestamp", "segment_id"] + FEATURE_COLUMNS].round(8).to_csv(out, index=False)
print("wrote", out, f"({len(featured):,} rows)")

### Two deliberate exclusions

`body_pct = (close - open) / close` is **not** a feature. In a 24/7 market
`open[t] == close[t-1]`, so it duplicates `return_1` (correlation > 0.999).

`buy_pressure = 2 * taker_buy_ratio - 1` is an affine duplicate of an existing
feature and is also excluded.

Both decisions, plus the full rationale for excluding absolute price levels,
are documented at the bottom of `src/features.py`.